# 02 — Baseline Models

Trains and cross-validates 20 classifiers on the prepared features.
Computes per-model segment disparity for the Contract attribute.
Generates **Figure 1** of the paper.

**Maps to paper Section 9.2.**


## 1. Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")


## 2. Load processed data from notebook 01

In [ ]:
DATA = Path("../outputs/tables")
X = pd.read_csv(DATA / "X_features.csv")
y = pd.read_csv(DATA / "y_target.csv").squeeze("columns")
segments_df = pd.read_csv(DATA / "segments.csv")
segments = segments_df["Contract"].values

print(f"X: {X.shape}, y: {y.shape}")
print(f"Segment values: {np.unique(segments)}")


## 3. Define the 20-model lineup

In [ ]:
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    AdaBoostClassifier, ExtraTreesClassifier,
)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.neural_network import MLPClassifier
import xgboost as xgb
import lightgbm as lgb

models = {
    "Logistic Regression (L2)": LogisticRegression(max_iter=2000, random_state=42),
    "Logistic Regression (L1)": LogisticRegression(penalty="l1", solver="saga", max_iter=2000, random_state=42),
    "Ridge Classifier":         RidgeClassifier(random_state=42),
    "Decision Tree":            DecisionTreeClassifier(random_state=42),
    "Random Forest":            RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "Extra Trees":              ExtraTreesClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "Gradient Boosting":        GradientBoostingClassifier(n_estimators=100, random_state=42),
    "AdaBoost":                 AdaBoostClassifier(n_estimators=100, random_state=42),
    "XGBoost":                  xgb.XGBClassifier(n_estimators=100, random_state=42, eval_metric="logloss", use_label_encoder=False),
    "LightGBM":                 lgb.LGBMClassifier(n_estimators=100, random_state=42, verbose=-1),
    "SVM (RBF)":                SVC(kernel="rbf", probability=True, random_state=42),
    "SVM (Linear)":             SVC(kernel="linear", probability=True, random_state=42),
    "k-NN (k=5)":               KNeighborsClassifier(n_neighbors=5),
    "k-NN (k=10)":              KNeighborsClassifier(n_neighbors=10),
    "Gaussian NB":              GaussianNB(),
    "Bernoulli NB":             BernoulliNB(),
    "MLP (2 layers)":           MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=500, random_state=42),
    "MLP (1 layer)":            MLPClassifier(hidden_layer_sizes=(100,), max_iter=500, random_state=42),
}
print(f"Defined {len(models)} models.")


## 4. Cross-validate global metrics

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, precision_score, recall_score, f1_score

def evaluate_cv(model, X, y, cv=5):
    scoring = {
        "accuracy": "accuracy",
        "precision": make_scorer(precision_score, zero_division=0),
        "recall":    make_scorer(recall_score, zero_division=0),
        "f1":        make_scorer(f1_score, zero_division=0),
    }
    res = cross_validate(
        model, X, y,
        cv=StratifiedKFold(n_splits=cv, shuffle=True, random_state=42),
        scoring=scoring, return_train_score=False,
    )
    return {m: float(np.mean(res[f"test_{m}"])) for m in scoring}

results_list = []
for name, model in models.items():
    print(f"  Evaluating {name}...")
    try:
        scores = evaluate_cv(model, X, y, cv=5)
        scores["Model"] = name
        results_list.append(scores)
    except Exception as e:
        print(f"    skipped ({e})")


## 5. Compute segment disparity per model

In [ ]:
from src.metrics import segment_disparity

disparities = {}
for name, model in models.items():
    print(f"  Disparity for {name}...")
    try:
        disparities[name] = segment_disparity(model, X.values, y.values, segments)
    except Exception:
        disparities[name] = np.nan


## 6. Assemble results table

In [ ]:
results_df = pd.DataFrame(results_list)
results_df = results_df[["Model", "accuracy", "precision", "recall", "f1"]]
results_df["disparity"] = results_df["Model"].map(disparities)
results_df = results_df.round(4).sort_values("accuracy", ascending=False).reset_index(drop=True)

results_df.to_csv("../outputs/tables/baseline_model_comparison.csv", index=False)
print(results_df.to_string(index=False))


## 7. Figure 1 — Accuracy vs Segment Disparity

In [ ]:
from src.visualizations import plot_accuracy_vs_disparity, savefig

fig = plot_accuracy_vs_disparity(results_df)
savefig(fig, "../outputs/figures/fig1_accuracy_vs_disparity.png")
fig.show()


## Key takeaway

High-accuracy models (Gradient Boosting, XGBoost, Random Forest) reach
~87-89% accuracy *while exhibiting 12-15% segment disparity*. This is
the central empirical finding of Section 9.2: global accuracy alone
can mask significant per-segment bias.

Continue with [03_ovb_demonstration.ipynb](./03_ovb_demonstration.ipynb).
